# 8. Encoding and scaling

Standalone notebook: it includes its own imports and runs top to bottom with no input prompts. Each `assert` raises an error if something is wrong, so no error means the section works.

In [ ]:
import numpy as np
import pandas as pd

## Label encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, RobustScaler

colors = ["Red", "Blue", "Green", "Red", "Blue"]

encoder = LabelEncoder()
encoded_colors = encoder.fit_transform(colors)

print("Original:", colors)
print("Encoded: ", encoded_colors)
print("Classes: ", list(encoder.classes_))

assert list(encoder.classes_) == ["Blue", "Green", "Red"]   # codes are alphabetical
assert list(encoded_colors) == [2, 0, 1, 2, 0]

## One-hot encoding

In [ ]:
df_colors = pd.DataFrame({"Colors": ["Red", "Blue", "Green", "Red", "Blue"]})
one_hot = pd.get_dummies(df_colors, columns=["Colors"], dtype=int)
print(one_hot)

assert list(one_hot.columns) == ["Colors_Blue", "Colors_Green", "Colors_Red"]
assert one_hot.sum(axis=1).eq(1).all()   # exactly one 1 per row

## Target (mean) encoding

Replace each category with the mean of the target for that category.

> Compute the means on training data only. Computing them on the full dataset lets information about
> the target leak into the features.

In [ ]:
def target_encode(df: pd.DataFrame, category_col: str, target_col: str):
    """Return (encoded_series, category_means)."""
    means = df.groupby(category_col)[target_col].mean()
    return df[category_col].map(means), means


df_houses = pd.DataFrame({
    "City": ["Lahore", "Lahore", "Lahore", "Karachi", "Karachi", "Karachi", "Islamabad", "Islamabad"],
    "Price_M": [10, 12, 11, 8, 9, 10, 15, 16],
})
df_houses["City_Target_Encoded"], city_means = target_encode(df_houses, "City", "Price_M")
print(city_means, "\n")
print(df_houses)

df_dep = pd.DataFrame({
    "Department": ["IT", "IT", "HR", "HR", "Sales", "Sales"],
    "Salary_K": [10, 12, 11, 8, 14, 10],
})
df_dep["Department_Target_Encoded"], salary_means = target_encode(df_dep, "Department", "Salary_K")
print("\n", salary_means, "\n")
print(df_dep)

assert city_means.to_dict() == {"Islamabad": 15.5, "Karachi": 9.0, "Lahore": 11.0}
assert salary_means.to_dict() == {"HR": 9.5, "IT": 11.0, "Sales": 12.0}

## Min-Max scaling

Rescales values to the range 0 to 1.

In [ ]:
ages = np.array([[20], [30], [40], [50], [60]])
ages_minmax = MinMaxScaler().fit_transform(ages)

print("Original:     ", ages.ravel())
print("Min-Max scaled:", ages_minmax.ravel())

assert np.allclose(ages_minmax.ravel(), [0, 0.25, 0.5, 0.75, 1])

## Robust scaling

Uses the median and IQR, so an extreme value (the 7000 below) does not distort the other values.

In [ ]:
salaries = np.array([[30], [35], [40], [42], [45], [50], [7000]])
salaries_robust = RobustScaler().fit_transform(salaries)

print("Original:     ", salaries.ravel())
print("Robust scaled:", np.round(salaries_robust.ravel(), 3))

assert abs(np.median(salaries_robust)) < 1e-9   # centred on the median

## Log transform (`log1p`)

`log1p(x)` computes `log(1 + x)`, which handles zeros and compresses large values.

In [ ]:
data = np.array([0, 10, 100, 1000, 10000])
log_data = np.log1p(data)
print(log_data)

assert log_data[0] == 0
assert np.allclose(log_data, np.log(data + 1))

## Decimal scaling

Divide by `10**j`, where `j` is the number of digits in the largest absolute value, so every result is below 1 in magnitude.

In [ ]:
def decimal_scaling(values: list):
    max_abs = max(abs(x) for x in values)
    j = len(str(int(max_abs)))
    return [x / (10 ** j) for x in values], j


scaled, j = decimal_scaling([-25, -50, -75, -100])
print("j =", j)
print(scaled)

assert j == 3
assert np.allclose(scaled, [-0.025, -0.05, -0.075, -0.1])
assert all(abs(x) < 1 for x in scaled)

In [ ]:
print("Encoding and scaling: all tests passed.")